In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import tensorflow_model_optimization as tfmot # [TFMOT] Nhúng thư viện tối ưu hóa
import numpy as np
import pandas as pd
import os
import glob
from PIL import Image

# ==========================================
# CẤU HÌNH THÔNG SỐ
# ==========================================
#! ĐỪNG ĐỤNG
IMG_SIZE = 32
NUM_CLASSES = 10
# ĐỤNG ĐƯỢC
EPOCHS = 100
BATCH_SIZE = 8
DROPOUT_RATE = 0.2
PATIENCE = 20
VAL_SPLIT = 0.3
DENSE = 64
OPTIMIZER = 'adam'

# DATA AUGMENTATION
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05, seed=123, fill_mode='nearest'),
    layers.RandomZoom(0.1, seed=123, fill_mode='nearest'),      
    layers.RandomTranslation(0.1, 0.1, seed=123, fill_mode='nearest'), 
    layers.RandomBrightness(factor=0.1, value_range=(0.0, 1.0), seed=123)
])

# ĐƯỜNG DẪN DATASET (Bạn sẽ cập nhật sau)
TRAIN_DIR = 'C:/DUT/Ki_2/Edge_AI/dataset/train'
TEST_DIR = 'C:/DUT/Ki_2/Edge_AI/dataset/test'

# ==========================================
# NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

def build_micro_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        
        layers.GlobalAveragePooling2D(),
        layers.Dense(DENSE, activation='relu'),
        layers.Dropout(DROPOUT_RATE),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

# Khởi tạo model gốc
base_model = build_micro_model()

# ==========================================
# [TFMOT] THIẾT LẬP PRUNING (CẮT TỈA TRỌNG SỐ)
# ==========================================
# Tính toán tổng số bước huấn luyện
steps_per_epoch = len(train_ds)
end_step = np.ceil(steps_per_epoch).astype(np.int32) * EPOCHS

# Cấu hình mức độ cắt tỉa: Bắt đầu từ 20% sparsity và tăng dần lên 75% ở epoch cuối
pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.20,
                                                            final_sparsity=0.75,
                                                            begin_step=0,
                                                            end_step=end_step)
}

# Wrap model gốc bằng hàm Pruning
model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(base_model, **pruning_params)

# ==========================================
# CUSTOM CALLBACK & COMPILE
# ==========================================
class AccuracyTracker(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best_val_acc = 0.0
        self.best_val_loss = 10.0

    def on_epoch_end(self, epoch, logs=None):
        current_val_acc = logs.get("val_accuracy")
        current_val_loss = logs.get("val_loss")
        if current_val_acc > self.best_val_acc:
            diff = current_val_acc - self.best_val_acc
            print(f"\n✅ Epoch {epoch+1}: Val-Accuracy tăng thêm {diff:.4f}! (Từ {self.best_val_acc:.4f} -> {current_val_acc:.4f})")
            self.best_val_acc = current_val_acc
        
        if current_val_loss < self.best_val_loss:
            diff_loss = self.best_val_loss - current_val_loss
            print(f"✅ Epoch {epoch+1}: Val-Loss giảm thêm {diff_loss:.4f}! (Từ {self.best_val_loss:.4f} -> {current_val_loss:.4f})")
            self.best_val_loss = current_val_loss

# Chú ý compile model_for_pruning, KHÔNG phải base_model
model_for_pruning.compile(optimizer=OPTIMIZER,
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            metrics=['accuracy'])

model_for_pruning.summary()

checkpoint = callbacks.ModelCheckpoint(
    'traffic_sign_model.h5', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss', patience=PATIENCE, restore_best_weights=True
)

# [TFMOT] Thêm UpdatePruningStep vào danh sách callbacks
callbacks_list = [
    AccuracyTracker(), 
    reduce_lr, 
    checkpoint, 
    early_stopping,
    tfmot.sparsity.keras.UpdatePruningStep() # Cập nhật mức độ Pruning sau mỗi step
]

# Tiến hành huấn luyện
history = model_for_pruning.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list
)

print("Đã lưu model Pruned: traffic_sign_model.h5")

# ==========================================
# [TFMOT] LỘT BỎ LỚP PRUNING VÀ XUẤT TFLITE
# ==========================================
# Load lại model tốt nhất (cần custom_objects để Keras hiểu lớp PruneLowMagnitude)
best_model = tf.keras.models.load_model(
    'traffic_sign_model.h5',
    custom_objects={'PruneLowMagnitude': tfmot.sparsity.keras.PruneLowMagnitude}
)

# Strip Pruning: Gỡ bỏ các wrapper dư thừa của tfmot, trả model về dạng chuẩn để convert
model_for_export = tfmot.sparsity.keras.strip_pruning(best_model)

# Khởi tạo TFLite Converter từ model đã được strip
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    for images, _ in train_ds.unbatch().batch(1).take(100):
        yield [images]

converter.representative_dataset = representative_dataset

# [TFMOT] Giữ nguyên ép kiểu INT8 (Bây giờ mô hình của bạn vừa Sparse vừa Quantized 8-bit)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open('traffic_sign_model_quantized_pruned.tflite', 'wb') as f:
    f.write(tflite_quant_model)

# ==========================================
# TEST TFLITE MODEL & XUẤT CSV
# ==========================================
# Load TFLite Model
interpreter = tf.lite.Interpreter(model_path="traffic_sign_model_quantized_pruned.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_scale, input_zero_point = input_details[0]['quantization']

test_images_paths = glob.glob(os.path.join(TEST_DIR, '*.png'))
test_images_paths.sort()

results = []

for img_path in test_images_paths:
    img_id = os.path.basename(img_path).split('.')[0]
    
    img = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    
    if input_scale != 0:
        img_array = img_array / input_scale + input_zero_point
    img_array = np.expand_dims(img_array.astype(np.int8), axis=0)
    
    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]
    
    predicted_class = np.argmax(output_data)
    results.append({'Id': img_id, 'Label': predicted_class})

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)

print("Đã xuất file: submission.csv")
print(f"Kích thước model TFLite (Int8 + Pruned): {len(tflite_quant_model) / 1024:.2f} KB")

# Note: Để thấy sức mạnh thực sự của Pruning, bạn có thể nén file tflite bằng ZIP. 
# File .tflite gốc có thể không giảm dung lượng, nhưng khi nén zip nó sẽ nhỏ đi 3-4 lần so với model INT8 bình thường!

Found 2000 files belonging to 10 classes.
Using 1400 files for training.
Found 2000 files belonging to 10 classes.
Using 600 files for validation.


c:\Users\ngong\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: `prune_low_magnitude` can only prune an object of the following types: keras.models.Sequential, keras functional model, keras.layers.Layer, list of keras.layers.Layer. You passed an object of type: Sequential.